# Manual Preprocessing & Baseline Forecasting (Ad-Hoc Workflow)
## Course: Data Science Tools and Techniques (DSTT)

Demonstrates how weather forecasting models are typically prepared today: ad-hoc scripts, global mean/forward-fill, manual lags, and raw features.

In [ ]:
import numpy as np
import pandas as pd
from weatherpipe.loader import load_raw_city_data
from weatherpipe.manual_baseline import prepare_manual_baseline_a_naive, prepare_manual_baseline_b_analyst
from weatherpipe.models import get_model
from weatherpipe.evaluation import compute_regression_metrics, generate_temporal_folds

### 1. Load Raw San Francisco Weather Records

In [ ]:
df_raw = load_raw_city_data('San Francisco', raw_dir='data/raw')
print('Raw shape:', df_raw.shape)

### 2. Prepare Manual Baselines
- **Baseline A (Naive)**: Column mean fill, raw columns
- **Baseline B (Analyst)**: Forward fill, calendar integers, 1 lag, 1 rolling mean

In [ ]:
X_naive, y_naive = prepare_manual_baseline_a_naive(df_raw, target_horizon=6)
X_analyst, y_analyst = prepare_manual_baseline_b_analyst(df_raw, target_horizon=6)
print('Analyst feature columns:', list(X_analyst.columns))

### 3. Evaluate Manual Baseline B on Temporal Fold

In [ ]:
folds = generate_temporal_folds(len(X_analyst), n_folds=4)
train_idx, test_idx = folds[-1]

model_analyst = get_model('xgboost')
mask_tr = y_analyst.iloc[train_idx].notna() & X_analyst.iloc[train_idx].notna().all(axis=1)
model_analyst.fit(X_analyst.iloc[train_idx][mask_tr], y_analyst.iloc[train_idx][mask_tr])

preds = model_analyst.predict(X_analyst.iloc[test_idx].fillna(0.0))
metrics = compute_regression_metrics(y_analyst.iloc[test_idx].values, preds)
print(f'Manual Baseline B (Analyst) Test RMSE: {metrics["rmse"]:.2f}°C, MAE: {metrics["mae"]:.2f}°C, R²: {metrics["r2"]:.4f}')